# 03 — Embedding and Index Verification

**Purpose:** Verify that the persisted FAISS index matches the legal chunks and configured embedding model.

Index construction lives in `laborlaw_rag.search`; rebuilding is opt-in because it calls the embedding API.


## 1. Setup


In [ ]:
from laborlaw_rag.config import Settings
from laborlaw_rag.data import load_chunks
from laborlaw_rag.search import FaissStore, build_vector_index
from laborlaw_rag.services import JinaClient

settings = Settings.from_env()
REBUILD_INDEX = False

## 2. Load Deployment Chunks


In [ ]:
chunks = load_chunks(settings.chunks_path, settings.source_url)
{"chunk_count": len(chunks), "chunks_path": str(settings.chunks_path)}

## 3. Rebuild or Load the Index

Keep `REBUILD_INDEX = False` for normal verification. Enable it only after the chunk artifact or embedding model changes.


In [ ]:
rebuild_manifest = None
if REBUILD_INDEX:
    rebuild_manifest = build_vector_index(chunks, JinaClient(settings), settings)

store = FaissStore.load(
    settings.index_path,
    chunks,
    chunks_path=settings.chunks_path,
    expected_model=settings.embedding_model,
    expected_document_task=settings.embedding_document_task,
    expected_query_task=settings.embedding_query_task,
)

## 4. Verify Compatibility


In [ ]:
{
    "status": "verified",
    "index_path": str(settings.index_path),
    "embedding_model": settings.embedding_model,
    "vector_count": store.index.ntotal,
    "dimension": store.index.d,
    "chunk_count": len(store.chunks),
    "rebuilt": REBUILD_INDEX,
    "rebuild_manifest": rebuild_manifest,
}